# Databricks + Lakehouse (batch)
**Objetivo:** demostrar carga y procesamiento batch en un data lake/lakehouse sobre Databricks usando datasets reales.

## Qué cubre este notebook
1. Cargar `.csv`
2. Cargar `.json`
3. Cargar `.parquet`
4. Convertir `.csv` a `.parquet`
5. Convertir a **Delta Lake** y probar funcionalidades
6. Operaciones sobre DataFrames: agregar/eliminar columnas, joins, filtros, agregaciones
7. EDA con Spark
8. SQL sobre Spark
9. Regresión lineal supervisada
10. K-Means
11. Ejemplos batch más avanzados: ventanas, deduplicación, medallion pattern, time travel

## Datasets reales usados
- **NYC Open Data – 311 Service Requests** en CSV y JSON
- **NYC TLC Yellow Taxi Trips** en Parquet

## Ambiente de ejecuón

- Diseñado para ejecutarse en un notebook de **Databricks** con runtime que ya incluya Apache Spark y Delta Lake.



In [0]:
# Parámetros base para que todo quede organizado en DBFS
base_path = "/Volumes/mycatalog262/myschema262/myvolume262"
raw_path = f"{base_path}/raw"
bronze_path = f"{base_path}/bronze"
silver_path = f"{base_path}/silver"
gold_path = f"{base_path}/gold"
ml_path = f"{base_path}/ml"

for p in [base_path, raw_path, bronze_path, silver_path, gold_path, ml_path]:
    dbutils.fs.mkdirs(p)

display(dbutils.fs.ls(bronze_path))

## 1) Descarga de datasets reales hacia DBFS

En Databricks, una forma práctica de trabajar con archivos remotos es descargarlos primero a `/dbfs/...`
y luego leerlos con Spark desde `dbfs:/...`.

In [0]:

import os
import urllib.request

os.makedirs(raw_path, exist_ok=True)

files = {
    # CSV real desde NYC Open Data (311)
    "311_sample.csv": "https://data.cityofnewyork.us/resource/erm2-nwe9.csv?$limit=50000",

    # JSON real desde NYC Open Data (311)
    "311_sample.json": "https://data.cityofnewyork.us/resource/erm2-nwe9.json?$limit=20000",

    # Parquet real desde NYC TLC
    "yellow_tripdata_2025-01.parquet": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-01.parquet"
}

for filename, url in files.items():
    target = os.path.join(raw_path, filename)
    if not os.path.exists(target):
        print(f"Descargando {filename} ...")
        urllib.request.urlretrieve(url, target)
    else:
        print(f"Ya existe: {filename}")

print("Archivos descargados en:", raw_path)

In [0]:

display(dbutils.fs.ls(raw_path))

## 2) Cargar archivos CSV

In [0]:
csv_path = f"{bronze_path}/flights/"

df_csv = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(csv_path)
)

print("Schema CSV:")
df_csv.printSchema()
display(df_csv.limit(10))

In [0]:
airports = spark.read.csv(f"{bronze_path}/aircraft/",
                          header=True,
                          inferSchema=True)
airports.printSchema()

## 3) Cargar archivos JSON

In [0]:
json_path = f"{raw_path}/311_sample.json"

df_json = (
    spark.read
    .option("multiline", True)
    .json(json_path)
)

print("Schema JSON:")
df_json.printSchema()
display(df_json.limit(10))

## 4) Cargar archivos Parquet

In [0]:

parquet_path = f"{raw_path}/yellow_tripdata_2025-01.parquet"

df_taxi = spark.read.parquet(parquet_path)

print("Schema Parquet:")
df_taxi.printSchema()
display(df_taxi.limit(10))

## 5) Convertir CSV a Parquet

Guardamos una copia del dataset 311 en Parquet para mostrar conversión de formato.

In [0]:

csv_to_parquet_path = f"{bronze_path}/311_parquet"

(
    df_csv.write
    .mode("overwrite")
    .parquet(csv_to_parquet_path)
)

df_311_parquet = spark.read.parquet(csv_to_parquet_path)

print("Filas en CSV original:", df_csv.count())
print("Filas en Parquet convertido:", df_311_parquet.count())
display(df_311_parquet.limit(10))

## 6) Operaciones de DataFrame
- seleccionar columnas
- crear columnas derivadas
- eliminar columnas
- filtrar
- agregar
- unir DataFrames

In [0]:
from pyspark.sql import functions as F

taxi_enriched = (
    df_taxi
    .withColumn("trip_duration_min",
                (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60.0)
    .withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .withColumn("speed_mph",
                F.when(F.col("trip_duration_min") > 0,
                       F.col("trip_distance") / (F.col("trip_duration_min") / 60.0)))
)

display(taxi_enriched.select(
    "VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime",
    "trip_distance", "fare_amount", "tip_amount",
    "trip_duration_min", "pickup_hour", "speed_mph"
).limit(10))

In [0]:

# Ejemplo de drop de columnas
taxi_reduced = taxi_enriched.drop("store_and_fwd_flag", "RatecodeID")
display(taxi_reduced.limit(5))

In [0]:

# Filtrado de calidad básica
taxi_clean = (
    taxi_enriched
    .filter(F.col("trip_distance") > 0)
    .filter(F.col("fare_amount") > 0)
    .filter(F.col("trip_duration_min") > 0)
    .filter(F.col("trip_duration_min") < 240)
    .filter(F.col("passenger_count").isNotNull())
)

print("Filas limpias:", taxi_clean.count())
display(taxi_clean.limit(10))

In [0]:

# Agregaciones
daily_stats = (
    taxi_clean.groupBy("pickup_date")
    .agg(
        F.count("*").alias("num_trips"),
        F.avg("trip_distance").alias("avg_trip_distance"),
        F.avg("fare_amount").alias("avg_fare"),
        F.avg("tip_amount").alias("avg_tip"),
        F.avg("trip_duration_min").alias("avg_duration_min")
    )
    .orderBy("pickup_date")
)

display(daily_stats)

In [0]:

# Join sencillo entre agregados por vendor y una dimensión pequeña creada en el notebook
vendor_dim = spark.createDataFrame(
    [
        (1, "Creative Mobile Technologies"),
        (2, "VeriFone / Curb")
    ],
    ["VendorID", "vendor_name"]
)

vendor_stats = (
    taxi_clean.groupBy("VendorID")
    .agg(
        F.count("*").alias("trips"),
        F.avg("fare_amount").alias("avg_fare"),
        F.avg("trip_distance").alias("avg_distance")
    )
)

vendor_stats_joined = vendor_stats.join(vendor_dim, on="VendorID", how="left")
display(vendor_stats_joined.orderBy("VendorID"))

## 7) EDA con Spark

In [0]:

# Conteo, tipos y nulos
from pyspark.sql import functions as F

eda_cols = ["passenger_count", "trip_distance", "fare_amount", "tip_amount", "total_amount", "trip_duration_min"]

null_summary = taxi_clean.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) for c in eda_cols
])

display(null_summary)

In [0]:

# Descriptivos
display(taxi_clean.select(eda_cols).describe())

In [0]:

# Cuantiles aproximados
quantiles = taxi_clean.approxQuantile(
    ["trip_distance", "fare_amount", "tip_amount", "trip_duration_min"],
    [0.25, 0.5, 0.75, 0.95],
    0.01
)
quantiles

In [0]:

# Distribución por hora
hourly_stats = (
    taxi_clean.groupBy("pickup_hour")
    .agg(
        F.count("*").alias("num_trips"),
        F.avg("fare_amount").alias("avg_fare"),
        F.avg("trip_distance").alias("avg_distance")
    )
    .orderBy("pickup_hour")
)

display(hourly_stats)

In [0]:

# Top zonas origen por volumen
pickup_zone_stats = (
    taxi_clean.groupBy("PULocationID")
    .agg(
        F.count("*").alias("num_trips"),
        F.avg("fare_amount").alias("avg_fare"),
        F.avg("tip_amount").alias("avg_tip")
    )
    .orderBy(F.desc("num_trips"))
)

display(pickup_zone_stats.limit(20))

## 8) SQL sobre Spark

In [0]:

taxi_clean.createOrReplaceTempView("taxi_clean")
df_csv.createOrReplaceTempView("service_requests_csv")

In [0]:

# Consulta SQL 1: resumen por hora
hourly_sql = spark.sql("""
SELECT
  hour(tpep_pickup_datetime) AS pickup_hour,
  COUNT(*) AS trips,
  ROUND(AVG(fare_amount), 2) AS avg_fare,
  ROUND(AVG(trip_distance), 2) AS avg_distance
FROM taxi_clean
GROUP BY hour(tpep_pickup_datetime)
ORDER BY pickup_hour
""")
display(hourly_sql)

In [0]:

# Consulta SQL 2: top categorías 311
requests_sql = spark.sql("""
SELECT complaint_type, COUNT(*) AS total
FROM service_requests_csv
WHERE complaint_type IS NOT NULL
GROUP BY complaint_type
ORDER BY total DESC
LIMIT 20
""")
display(requests_sql)

## 9) Convertir a Delta Lake y probar funcionalidades

Aquí construimos una tabla Delta a partir del dataset de taxis filtrado.

In [0]:

delta_taxi_path = f"{silver_path}/taxi_delta"

(
    taxi_clean.write
    .format("delta")
    .mode("overwrite")
    .save(delta_taxi_path)
)

df_delta = spark.read.format("delta").load(delta_taxi_path)
display(df_delta.limit(10))

In [0]:
# Registrar como vista temporal para consultas SQL
spark.sql("DROP VIEW IF EXISTS demo_taxi_delta")

spark.sql(f"""
CREATE OR REPLACE TEMP VIEW demo_taxi_delta
AS SELECT * FROM delta.`{delta_taxi_path}`
""")

display(spark.sql("SELECT COUNT(*) AS total_rows FROM demo_taxi_delta"))

In [0]:
# Historial de Delta
display(spark.sql(f"DESCRIBE HISTORY delta.`{delta_taxi_path}`"))

In [0]:
# UPDATE y DELETE en Delta con SQL
spark.sql(f"""
UPDATE delta.`{delta_taxi_path}`
SET fare_amount = fare_amount * 1.02
WHERE VendorID = 1 AND fare_amount < 20
""")

spark.sql(f"""
DELETE FROM delta.`{delta_taxi_path}`
WHERE trip_distance <= 0
""")

display(spark.sql(f"DESCRIBE HISTORY delta.`{delta_taxi_path}`"))

In [0]:
# MERGE INTO con datos de ejemplo
updates_df = spark.createDataFrame(
    [
        (1, "promo_applied"),
        (2, "standard")
    ],
    ["VendorID", "pricing_tag"]
)

updates_df.createOrReplaceTempView("vendor_updates")

spark.sql("DROP TABLE IF EXISTS vendor_updates_tbl")
spark.sql("CREATE TABLE vendor_updates_tbl USING DELTA AS SELECT * FROM vendor_updates")

spark.sql(f"""
MERGE INTO delta.`{delta_taxi_path}` AS t
USING vendor_updates_tbl AS s
ON t.VendorID = s.VendorID
WHEN MATCHED THEN
  UPDATE SET t.congestion_surcharge = coalesce(t.congestion_surcharge, 0.0)
""")

display(spark.sql(f"DESCRIBE HISTORY delta.`{delta_taxi_path}`"))

In [0]:
# Time travel: leer una versión anterior, si existe
history = spark.sql(f"DESCRIBE HISTORY delta.`{delta_taxi_path}`").collect()
versions = [row["version"] for row in history]

print("Versiones disponibles:", versions)

if len(versions) >= 2:
    older_version = sorted(versions)[-2]
    print("Leyendo versión anterior:", older_version)
    older_df = spark.read.format("delta").option("versionAsOf", older_version).load(delta_taxi_path)
    display(older_df.limit(10))
else:
    print("Aún no hay suficientes versiones para time travel.")

# Time travel: leer una versión anterior, si existe
history = spark.sql(f"DESCRIBE HISTORY delta.`{delta_taxi_path}`").collect()
versions = [row["version"] for row in history]

print("Versiones disponibles:", versions)

if len(versions) >= 2:
    older_version = sorted(versions)[-2]
    print("Leyendo versión anterior:", older_version)
    older_df = spark.read.format("delta").option("versionAsOf", older_version).load(delta_taxi_path)
    display(older_df.limit(10))
else:
    print("Aún no hay suficientes versiones para time travel.")

In [0]:

# Bronze: 311 crudo en Delta
bronze_311_path = f"{bronze_path}/311_delta"

(
    df_csv.write
    .format("delta")
    .mode("overwrite")
    .save(bronze_311_path)
)

# Silver: normalización mínima
df_311_silver = (
    spark.read.format("delta").load(bronze_311_path)
    .withColumn("created_date_ts", F.to_timestamp("created_date"))
    .withColumn("closed_date_ts", F.to_timestamp("closed_date"))
    .withColumn("resolution_days",
                (F.col("closed_date_ts").cast("long") - F.col("created_date_ts").cast("long")) / 86400.0)
)

silver_311_path = f"{silver_path}/311_delta"
df_311_silver.write.format("delta").mode("overwrite").save(silver_311_path)

# Gold: agregado por borough y complaint_type
df_311_gold = (
    spark.read.format("delta").load(silver_311_path)
    .groupBy("borough", "complaint_type")
    .agg(F.count("*").alias("num_requests"),
         F.avg("resolution_days").alias("avg_resolution_days"))
)

gold_311_path = f"{gold_path}/311_agg_delta"
df_311_gold.write.format("delta").mode("overwrite").save(gold_311_path)

display(spark.read.format("delta").load(gold_311_path).orderBy(F.desc("num_requests")).limit(20))

## 11) Ejemplos batch avanzados: ventanas, rankings, deduplicación

In [0]:

from pyspark.sql.window import Window

# Ranking de zonas por recaudo dentro de cada vendor
zone_revenue = (
    taxi_clean.groupBy("VendorID", "PULocationID")
    .agg(F.sum("total_amount").alias("revenue"))
)

w = Window.partitionBy("VendorID").orderBy(F.desc("revenue"))

zone_ranked = (
    zone_revenue
    .withColumn("rank_in_vendor", F.row_number().over(w))
)

display(zone_ranked.filter(F.col("rank_in_vendor") <= 10).orderBy("VendorID", "rank_in_vendor"))

In [0]:

# Detección y eliminación de duplicados en 311 (ejemplo batch)
dedup_keys = ["flight_id"]

df_311_dedup = (
    spark.read.format("delta").load(bronze_311_path)
    .dropDuplicates(dedup_keys)
)

print("Bronze filas:", spark.read.format("delta").load(bronze_311_path).count())
print("Sin duplicados:", df_311_dedup.count())

In [0]:
# Cache / persist para acelerar reutilización en análisis batch
# Nota: .cache() no está soportado en serverless compute
print("Filas:", taxi_clean.count())

## 12) ML supervisado: regresión lineal

Ejemplo sencillo: predecir `tip_amount` a partir de variables operativas del viaje.

In [0]:

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator

lr_df = (
    taxi_clean
    .select(
        "passenger_count",
        "trip_distance",
        "fare_amount",
        "trip_duration_min",
        "tip_amount"
    )
    .dropna()
    .filter(F.col("tip_amount") >= 0)
)

train_df, test_df = lr_df.randomSplit([0.8, 0.2], seed=42)

assembler = VectorAssembler(
    inputCols=["passenger_count", "trip_distance", "fare_amount", "trip_duration_min"],
    outputCol="features"
)

lr = LinearRegression(
    featuresCol="features",
    labelCol="tip_amount",
    predictionCol="prediction",
    maxIter=20,
    regParam=0.1,
    elasticNetParam=0.0
)

pipeline = Pipeline(stages=[assembler, lr])
lr_model = pipeline.fit(train_df)

pred_lr = lr_model.transform(test_df)

display(pred_lr.select("tip_amount", "prediction").limit(20))

evaluator_rmse = RegressionEvaluator(labelCol="tip_amount", predictionCol="prediction", metricName="rmse")
evaluator_r2 = RegressionEvaluator(labelCol="tip_amount", predictionCol="prediction", metricName="r2")

print("RMSE:", evaluator_rmse.evaluate(pred_lr))
print("R2:", evaluator_r2.evaluate(pred_lr))

## 13) K-Means

Clustering sencillo de viajes según distancia, tarifa y duración.

In [0]:

from pyspark.ml.clustering import KMeans
from pyspark.ml.feature import StandardScaler

km_df = (
    taxi_clean
    .select("trip_distance", "fare_amount", "trip_duration_min")
    .dropna()
)

train_km, test_km = km_df.randomSplit([0.8, 0.2], seed=42)

assembler_km = VectorAssembler(
    inputCols=["trip_distance", "fare_amount", "trip_duration_min"],
    outputCol="features_raw"
)

scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withMean=True,
    withStd=True
)

kmeans = KMeans(
    k=4,
    seed=42,
    featuresCol="features",
    predictionCol="cluster"
)

km_pipeline = Pipeline(stages=[assembler_km, scaler, kmeans])
km_model = km_pipeline.fit(train_km)

clustered = km_model.transform(test_km)
display(clustered.limit(20))

In [0]:

# Tamaños de cluster
display(
    clustered.groupBy("cluster")
    .agg(
        F.count("*").alias("n"),
        F.avg("trip_distance").alias("avg_distance"),
        F.avg("fare_amount").alias("avg_fare"),
        F.avg("trip_duration_min").alias("avg_duration")
    )
    .orderBy("cluster")
)

## 14) SQL directo sobre Delta

In [0]:

spark.sql("DROP VIEW IF EXISTS gold_requests")

spark.sql(f"""
CREATE OR REPLACE TEMP VIEW gold_requests
AS SELECT * FROM delta.`{gold_311_path}`
""")

display(spark.sql("""
SELECT borough, SUM(num_requests) AS total_requests
FROM gold_requests
GROUP BY borough
ORDER BY total_requests DESC
"""))

## 15) Opcionales Databricks-only

Estas celdas son útiles si su workspace permite comandos de mantenimiento como `OPTIMIZE` y `VACUUM`.
Descomente si quiere usarlas.

In [0]:

# spark.sql("OPTIMIZE demo_taxi_delta ZORDER BY (VendorID, PULocationID)")
# spark.sql("VACUUM demo_taxi_delta RETAIN 168 HOURS")

## 16) Limpieza opcional

In [0]:

# Descomente si desea borrar todo el laboratorio
# dbutils.fs.rm(base_path, True)